# ChaseEscapeEnv – 3 models, team rewards, strategy planner (Colab)

**Goal:** Police + enemy + **strategy planner** (3 agents). Strategy coordinates police: go to place, trap, bottleneck, block exit. **Team shared rewards**; supports multi police/enemy later (1v1 first). Three zips for higher accuracy.

**Training uses random maps and random spawns** so the agent sees different walls and police/enemy positions every episode. This improves generalization and helps learning. Each step always trains a **new** model from scratch (no loading previous zips).

**Step 1 – Train police (enemy static)**  
Random walls + random positions each episode. Enemy does **not** move. Only police is trained to find and arrest. → `ppo_chase_escape_final.zip`.

**Step 2 – Train enemy (police = trained model)**  
Load police zip; train enemy from scratch with random maps + random positions. → `ppo_enemy_escape_final.zip`.

**Step 3 – Train strategy planner (3rd model)**  
Random maps + random positions. Strategy commands police; enemy scripted or loaded. Train from scratch. → `ppo_strategy_final.zip`.

**Run all 3:** Strategy + enemy models; optional map (e.g. training_map.json) for evaluation.

In [ ]:
# Clone the pursuit_arena repository in Colab and move into the project folder.
# This uses your GitHub repo: https://github.com/GraslyDias/pursuit_arena.git

%cd /content
!git clone https://github.com/GraslyDias/pursuit_arena.git
%cd /content/pursuit_arena

In [ ]:
# Install dependencies in Colab only (NOT on your local machine).

!pip install pygame gymnasium stable-baselines3[extra] numpy

In [ ]:
# Optional: quick environment check
from pursuit_arena.ai.rl.chase_escape_env import ChaseEscapeEnv
from stable_baselines3.common.env_checker import check_env

env = ChaseEscapeEnv(render_mode=None)
check_env(env, warn=True)
env.close()

### Why we use CPU for training (and when GPU is faster)

- **This project uses `MlpPolicy`** (small fully connected network) and **vector observations** (a list of numbers), not images.
- **GPU is best at:** huge parallel math (e.g. millions of pixels in a CNN, big matrix multiplies). Our policy is small, so the GPU has little work and is underused.
- **Where time goes:** Most of each step is spent **running the environment** (Python, physics, collisions) on the **CPU**. The neural network forward/backward pass is a small part. So the bottleneck is CPU env, not GPU compute.
- **Data transfer:** Observations and actions are produced on CPU. Using GPU means copying data CPU→GPU and back. For small batches and small nets, that copy cost can make GPU **slower** than CPU.
- **When to use GPU:** Use GPU (e.g. `device="cuda"`) when you use **CNN policies** (learning from pixels/images) or **very large** networks. Then the GPU has enough parallel work to be faster.
- **Summary:** For our setup, CPU is typically **as fast or faster**. If you remove `device="cpu"`, PPO will use GPU if available but you may see a warning and similar or worse speed.

In [ ]:
# Step 1: Train POLICE from scratch. Random walls + random spawn positions every episode (no saved map).
from pathlib import Path

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

from pursuit_arena.ai.rl.chase_escape_env import ChaseEscapeEnv

log_dir = Path("runs/ppo_chase_escape")
log_dir.mkdir(parents=True, exist_ok=True)

def make_env():
    env = ChaseEscapeEnv(render_mode=None)
    env.static_enemy = True   # Enemy does not move; only police is trained to find and arrest
    # No training_map_options: every reset() uses random walls + random police/enemy positions
    env = Monitor(env, str(log_dir / "monitor.csv"))
    return env

vec_env = DummyVecEnv([make_env])

# Always train a new model from scratch (no loading existing zip)
model_path = log_dir / "ppo_chase_escape_final.zip"
model = PPO("MlpPolicy", vec_env, verbose=1, tensorboard_log=str(log_dir / "tb"), device="cpu")
print("Training police from scratch with random maps and random spawns.")
model.learn(total_timesteps=300_000)
model.save(str(model_path))
print("Saved model to", model_path)
vec_env.close()

## Step 2: Train **enemy** from scratch (police = trained model)

Run this **after** Step 1. Police is the **trained** model (loaded from zip). Enemy is trained from scratch with **random walls and random spawn positions** every episode. Saves `ppo_enemy_escape_final.zip`.

In [ ]:
# Step 2: Train ENEMY from scratch. Police = trained model. Random walls + random spawns every episode.
from pathlib import Path

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

from pursuit_arena.ai.rl.chase_escape_env import ChaseEscapeEnemyEnv

log_dir = Path("runs/ppo_chase_escape")
log_dir.mkdir(parents=True, exist_ok=True)

# Load trained police model (opponent; do not train it)
police_zip = log_dir / "ppo_chase_escape_final.zip"
assert police_zip.exists(), "Run Step 1 first to create ppo_chase_escape_final.zip"
police_model = PPO.load(str(police_zip), device="cpu")

def make_env_enemy():
    env = ChaseEscapeEnemyEnv(render_mode=None, police_model=police_model)
    # No training_map_options: random walls + random positions each episode
    env = Monitor(env, str(log_dir / "monitor_enemy.csv"))
    return env

vec_env = DummyVecEnv([make_env_enemy])

# Always train a new enemy model from scratch
model_path_enemy = log_dir / "ppo_enemy_escape_final.zip"
model_enemy = PPO("MlpPolicy", vec_env, verbose=1, tensorboard_log=str(log_dir / "tb_enemy"), device="cpu")
print("Training enemy from scratch with random maps and random spawns.")
model_enemy.learn(total_timesteps=300_000)
model_enemy.save(str(model_path_enemy))
print("Saved enemy model to", model_path_enemy)
vec_env.close()

## Step 3: Train strategy planner from scratch (3rd model)

Strategy planner is the RL agent. Random walls + random spawns every episode. Commands police: chase, block exit, bottleneck, hold, flank. **Team shared reward** (arrest +, escape −). Enemy scripted or trained model. Saves `ppo_strategy_final.zip`. Run after Step 1 and Step 2.

In [ ]:
# Step 3: Train STRATEGY PLANNER from scratch. Random walls + random spawns. Team shared rewards.
from pathlib import Path

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

from pursuit_arena.ai.rl.chase_escape_env import ChaseEscapeStrategyEnv

log_dir = Path("runs/ppo_chase_escape")
log_dir.mkdir(parents=True, exist_ok=True)

# Optional: use trained enemy so strategy learns vs strong opponent
enemy_zip = log_dir / "ppo_enemy_escape_final.zip"
enemy_model = PPO.load(str(enemy_zip), device="cpu") if enemy_zip.exists() else None

def make_env_strategy():
    env = ChaseEscapeStrategyEnv(render_mode=None, enemy_model=enemy_model)
    # No training_map_options: random walls + random positions each episode
    env = Monitor(env, str(log_dir / "monitor_strategy.csv"))
    return env

vec_env = DummyVecEnv([make_env_strategy])
model_path_strategy = log_dir / "ppo_strategy_final.zip"
# Always train a new strategy model from scratch
model_strategy = PPO("MlpPolicy", vec_env, verbose=1, tensorboard_log=str(log_dir / "tb_strategy"), device="cpu")
print("Training strategy from scratch with random maps and random spawns.")
model_strategy.learn(total_timesteps=300_000)
model_strategy.save(str(model_path_strategy))
print("Saved strategy model to", model_path_strategy)
vec_env.close()

## Run all 3 models (strategy commands police, enemy uses trained model)

Strategy planner sees state and outputs commands; police move according to strategy. Enemy uses trained enemy model. No training – evaluation only.

In [ ]:
# Run all 3: strategy + enemy models (strategy commands police)
from pathlib import Path

from stable_baselines3 import PPO

from pursuit_arena.ai.rl.chase_escape_env import ChaseEscapeStrategyEnv, load_training_map

log_dir = Path("runs/ppo_chase_escape")
options = load_training_map("training_map.json") if Path("training_map.json").exists() else None
enemy_model = PPO.load(str(log_dir / "ppo_enemy_escape_final.zip"), device="cpu") if (log_dir / "ppo_enemy_escape_final.zip").exists() else None
strategy_model = PPO.load(str(log_dir / "ppo_strategy_final.zip"), device="cpu")

env = ChaseEscapeStrategyEnv(render_mode=None, enemy_model=enemy_model)
if options is not None:
    env.training_map_options = options
    print("Using map: training_map.json")

episodes = 5
for ep in range(episodes):
    obs, _ = env.reset()
    done, truncated = False, False
    ep_rew = 0.0
    while not (done or truncated):
        action, _ = strategy_model.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = env.step(int(action))
        ep_rew += reward
    print(f"Episode {ep+1}: reward={ep_rew:.2f}, {info}")
env.close()

In [ ]:
# Evaluate the trained model in Colab (no rendering window)
from stable_baselines3 import PPO
from pursuit_arena.ai.rl.chase_escape_env import ChaseEscapeEnv

env = ChaseEscapeEnv(render_mode=None)
model = PPO.load("runs/ppo_chase_escape/ppo_chase_escape_final.zip", env=env)

episodes = 5
for ep in range(episodes):
    obs, info = env.reset()
    done = False
    truncated = False
    ep_reward = 0.0
    while not (done or truncated):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = env.step(action)
        ep_reward += float(reward)
    print(f"Episode {ep+1}: reward={ep_reward:.2f}, info={info}")

env.close()

In [ ]:
# Download trained models to your local machine (from Colab)
from google.colab import files

files.download("runs/ppo_chase_escape/ppo_chase_escape_final.zip")   # police
files.download("runs/ppo_chase_escape/ppo_enemy_escape_final.zip")   # enemy
files.download("runs/ppo_chase_escape/ppo_strategy_final.zip")      # strategy planner

## All-in-one continuous training (POLICE + ENEMY + STRATEGY)

Use this section when you want to **keep improving the same 3 models day by day**.

- If a zip already exists, it will be **loaded and continued**.
- If no zip exists yet, that model is trained **from scratch**.

This gives you one cell to run each day to improve all agents.

In [ ]:
# All-in-one trainer: POLICE -> ENEMY -> STRATEGY (continue from zips if they exist)
from pathlib import Path

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

from pursuit_arena.ai.rl.chase_escape_env import (
    ChaseEscapeEnv,
    ChaseEscapeEnemyEnv,
    ChaseEscapeStrategyEnv,
)


log_dir = Path("runs/ppo_chase_escape")
log_dir.mkdir(parents=True, exist_ok=True)


def train_police(total_timesteps: int = 200_000):
    def make_env():
        env = ChaseEscapeEnv(render_mode=None)
        env.static_enemy = True  # enemy does not move
        env = Monitor(env, str(log_dir / "monitor_police.csv"))
        return env

    vec_env = DummyVecEnv([make_env])

    model_path = log_dir / "ppo_chase_escape_final.zip"
    if model_path.exists():
        model = PPO.load(str(model_path), env=vec_env, device="cpu")
        print("POLICE: loaded existing model, continuing training.")
    else:
        model = PPO("MlpPolicy", vec_env, verbose=1, tensorboard_log=str(log_dir / "tb_police"), device="cpu")
        print("POLICE: training from scratch.")

    model.learn(total_timesteps=total_timesteps)
    model.save(str(model_path))
    print("POLICE: saved to", model_path)
    vec_env.close()


def train_enemy(total_timesteps: int = 200_000):
    police_path = log_dir / "ppo_chase_escape_final.zip"
    assert police_path.exists(), "Train POLICE first to create ppo_chase_escape_final.zip"
    police_model = PPO.load(str(police_path), device="cpu")

    def make_env():
        env = ChaseEscapeEnemyEnv(render_mode=None, police_model=police_model)
        env = Monitor(env, str(log_dir / "monitor_enemy.csv"))
        return env

    vec_env = DummyVecEnv([make_env])

    model_path = log_dir / "ppo_enemy_escape_final.zip"
    if model_path.exists():
        model = PPO.load(str(model_path), env=vec_env, device="cpu")
        print("ENEMY: loaded existing model, continuing training.")
    else:
        model = PPO("MlpPolicy", vec_env, verbose=1, tensorboard_log=str(log_dir / "tb_enemy"), device="cpu")
        print("ENEMY: training from scratch.")

    model.learn(total_timesteps=total_timesteps)
    model.save(str(model_path))
    print("ENEMY: saved to", model_path)
    vec_env.close()


def train_strategy(total_timesteps: int = 200_000):
    enemy_path = log_dir / "ppo_enemy_escape_final.zip"
    enemy_model = PPO.load(str(enemy_path), device="cpu") if enemy_path.exists() else None
    if enemy_model is None:
        print("STRATEGY: no enemy zip found, training vs scripted enemy.")
    else:
        print("STRATEGY: training vs trained enemy model.")

    def make_env():
        env = ChaseEscapeStrategyEnv(render_mode=None, enemy_model=enemy_model)
        env = Monitor(env, str(log_dir / "monitor_strategy.csv"))
        return env

    vec_env = DummyVecEnv([make_env])

    model_path = log_dir / "ppo_strategy_final.zip"
    if model_path.exists():
        model = PPO.load(str(model_path), env=vec_env, device="cpu")
        print("STRATEGY: loaded existing model, continuing training.")
    else:
        model = PPO("MlpPolicy", vec_env, verbose=1, tensorboard_log=str(log_dir / "tb_strategy"), device="cpu")
        print("STRATEGY: training from scratch.")

    model.learn(total_timesteps=total_timesteps)
    model.save(str(model_path))
    print("STRATEGY: saved to", model_path)
    vec_env.close()


# ---- Run the full 3-model training cycle ----
# You can run this cell many times (e.g. each day).
# Increase/decrease total_timesteps as you like.
train_police(total_timesteps=200_000)
train_enemy(total_timesteps=200_000)
train_strategy(total_timesteps=200_000)

In [ ]:
from google.colab import files

files.download("runs/ppo_chase_escape/ppo_chase_escape_final.zip")   # police
files.download("runs/ppo_chase_escape/ppo_enemy_escape_final.zip")   # enemy
files.download("runs/ppo_chase_escape/ppo_strategy_final.zip")      # strategy planner

## Reward shaping and wall penalties

This notebook uses **shaped rewards** so agents learn better paths:

- **Police (Step 1 / `ChaseEscapeEnv`)**:
  - `+5.0` when arresting the enemy.
  - `-10.0` when the enemy escapes.
  - `+0.05` per step when the police **sees** the enemy (inside FOV and LOS).
  - `-0.05` when moving forward but staying almost in the same position **inside the map** (hits wall / stuck).
  - `-0.01` when the police **barely moves at all** (discourages spinning in place).
  - `-0.002` small step penalty; `-1.0` when timeout with no arrest/escape.

- **Enemy (Step 2 / `ChaseEscapeEnemyEnv`)**:
  - `+5.0` when escaping off the map.
  - `-5.0` when being arrested.
  - `-0.05` when `enemy.blocked_last_step` is true (enemy tried to move but hit a wall and stayed in place).
  - `-0.002` step penalty; `-0.5` when timeout with no arrest/escape.

- **Strategy planner (Step 3 / `ChaseEscapeStrategyEnv`)**:
  - `+5.0` when police arrests the enemy (team success).
  - `-10.0` when the enemy escapes (team failure).
  - `-0.05` when the chosen strategy makes police try to move but stay stuck at a wall (except explicit HOLD).
  - `-0.002` step penalty; `-0.5` when timeout with no arrest/escape.

These penalties encourage agents to **avoid walls**, not get stuck, and learn smoother paths and strategies.

In [ ]:
# All-in-one continuous trainer with toggles (POLICE / ENEMY / STRATEGY)
from pathlib import Path

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

import pursuit_arena.ai.rl.chase_escape_env as ce
from pursuit_arena.ai.rl.chase_escape_env import (
    ChaseEscapeEnv,
    ChaseEscapeEnemyEnv,
    ChaseEscapeStrategyEnv,
    load_training_map,
)

# ---- Reward configuration (edit these to customize rewards) ----
# Police rewards
ce.POLICE_REWARD_ARREST = 5.0
ce.POLICE_REWARD_ESCAPE = -10.0
ce.POLICE_REWARD_VISIBLE = 0.05
ce.POLICE_PENALTY_WALL_COLLISION = -0.05
ce.POLICE_PENALTY_SPIN = -0.01
ce.POLICE_STEP_PENALTY = -0.002
ce.POLICE_TIMEOUT_PENALTY = -1.0

# Enemy rewards
ce.ENEMY_REWARD_ESCAPE = 5.0
ce.ENEMY_REWARD_ARREST = -5.0
ce.ENEMY_PENALTY_WALL_BLOCK = -0.05
ce.ENEMY_STEP_PENALTY = -0.002
ce.ENEMY_TIMEOUT_PENALTY = -0.5

# Strategy rewards
ce.STRATEGY_REWARD_ARREST = 5.0
ce.STRATEGY_REWARD_ESCAPE = -10.0
ce.STRATEGY_PENALTY_WALL_STUCK = -0.05
ce.STRATEGY_STEP_PENALTY = -0.002
ce.STRATEGY_TIMEOUT_PENALTY = -0.5

# ---- Toggle which models to train in this run ----
TRAIN_POLICE = True      # set False to skip police this run
TRAIN_ENEMY = True       # set False to skip enemy this run
TRAIN_STRATEGY = True    # set False to skip strategy this run

# Timesteps per model for this run
TIMESTEPS_POLICE = 200_000
TIMESTEPS_ENEMY = 200_000
TIMESTEPS_STRATEGY = 200_000

# Use saved training_map.json instead of random maps?
USE_TRAINING_MAP = True

log_dir = Path("runs/ppo_chase_escape")
log_dir.mkdir(parents=True, exist_ok=True)


def train_police(total_timesteps: int = 200_000):
    def make_env():
        env = ChaseEscapeEnv(render_mode=None)
        env.static_enemy = True  # enemy does not move
        if USE_TRAINING_MAP:
            map_path = Path("training_map.json")
            if map_path.exists():
                env.training_map_options = load_training_map(map_path)
                print("POLICE: using training_map.json")
        env = Monitor(env, str(log_dir / "monitor_police.csv"))
        return env

    vec_env = DummyVecEnv([make_env])

    model_path = log_dir / "ppo_chase_escape_final.zip"
    if model_path.exists():
        model = PPO.load(str(model_path), env=vec_env, device="cpu")
        print("POLICE: loaded existing model, continuing training.")
    else:
        model = PPO("MlpPolicy", vec_env, verbose=1, tensorboard_log=str(log_dir / "tb_police"), device="cpu")
        print("POLICE: training from scratch.")

    model.learn(total_timesteps=total_timesteps)
    model.save(str(model_path))
    print("POLICE: saved to", model_path)
    vec_env.close()


def train_enemy(total_timesteps: int = 200_000):
    police_path = log_dir / "ppo_chase_escape_final.zip"
    assert police_path.exists(), "Train POLICE first to create ppo_chase_escape_final.zip"
    police_model = PPO.load(str(police_path), device="cpu")

    def make_env():
        env = ChaseEscapeEnemyEnv(render_mode=None, police_model=police_model)
        if USE_TRAINING_MAP:
            map_path = Path("training_map.json")
            if map_path.exists():
                env.training_map_options = load_training_map(map_path)
                print("ENEMY: using training_map.json")
        env = Monitor(env, str(log_dir / "monitor_enemy.csv"))
        return env

    vec_env = DummyVecEnv([make_env])

    model_path = log_dir / "ppo_enemy_escape_final.zip"
    if model_path.exists():
        model = PPO.load(str(model_path), env=vec_env, device="cpu")
        print("ENEMY: loaded existing model, continuing training.")
    else:
        model = PPO("MlpPolicy", vec_env, verbose=1, tensorboard_log=str(log_dir / "tb_enemy"), device="cpu")
        print("ENEMY: training from scratch.")

    model.learn(total_timesteps=total_timesteps)
    model.save(str(model_path))
    print("ENEMY: saved to", model_path)
    vec_env.close()


def train_strategy(total_timesteps: int = 200_000):
    enemy_path = log_dir / "ppo_enemy_escape_final.zip"
    enemy_model = PPO.load(str(enemy_path), device="cpu") if enemy_path.exists() else None
    if enemy_model is None:
        print("STRATEGY: no enemy zip found, training vs scripted enemy.")
    else:
        print("STRATEGY: training vs trained enemy model.")

    def make_env():
        env = ChaseEscapeStrategyEnv(render_mode=None, enemy_model=enemy_model)
        if USE_TRAINING_MAP:
            map_path = Path("training_map.json")
            if map_path.exists():
                env.training_map_options = load_training_map(map_path)
                print("STRATEGY: using training_map.json")
        env = Monitor(env, str(log_dir / "monitor_strategy.csv"))
        return env

    vec_env = DummyVecEnv([make_env])

    model_path = log_dir / "ppo_strategy_final.zip"
    if model_path.exists():
        model = PPO.load(str(model_path), env=vec_env, device="cpu")
        print("STRATEGY: loaded existing model, continuing training.")
    else:
        model = PPO("MlpPolicy", vec_env, verbose=1, tensorboard_log=str(log_dir / "tb_strategy"), device="cpu")
        print("STRATEGY: training from scratch.")

    model.learn(total_timesteps=total_timesteps)
    model.save(str(model_path))
    print("STRATEGY: saved to", model_path)
    vec_env.close()


# ---- Run training according to toggles ----
if TRAIN_POLICE:
    train_police(total_timesteps=TIMESTEPS_POLICE)

if TRAIN_ENEMY:
    train_enemy(total_timesteps=TIMESTEPS_ENEMY)

if TRAIN_STRATEGY:
    train_strategy(total_timesteps=TIMESTEPS_STRATEGY)